In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from datetime import datetime

# ===================================
# PARAMETER
# ===================================

THEMES = {
    "QQQ": "BigTech",
    "SOXX": "Semiconductors",
    "AIQ": "Artificial_Intelligence",
    "ARKK": "Disruptive_Innovation",
    "SKYY": "Cloud_Computing"
}

TRAIN_START = "2024-01-01"
TRAIN_END = "2025-10-31"
EVENT_START = "2025-11-01"
EVENT_END = datetime.today().strftime("%Y-%m-%d")

MIN_SCORE = 4

tickers = tuple(sorted(set([
    'AAPL','MSFT','AMZN','GOOGL','META','NVDA','AVGO','TSLA','AMD','QCOM',
    'INTC','ADBE','NFLX','PEP','COST','TXN','AMGN','INTU','AMAT','LRCX',
    'ADI','ADP','PANW','REGN','SNPS','MU','MELI','KLAC','CDNS','CRWD',
    'PYPL','ORLY','NXPI','MNST','FTNT','MCHP','CTAS','FAST','BIIB',
    'IDXX','DDOG','ZS','TEAM','MRVL','NOW'
])))

# ===================================
# DATEN LADEN
# ===================================

all_assets = tickers + tuple(THEMES.keys())

data = yf.download(
    all_assets,
    start=TRAIN_START,
    end=EVENT_END,
    auto_adjust=True,
    progress=False
)

close = data["Close"]
volume = data["Volume"]

results = {}

# ===================================
# LOOP: JE AKTIE x JE THEMA
# ===================================

for stock in tickers:
    stock_ret = close[stock].pct_change().dropna()

    if len(stock_ret) < 250:
        continue

    for theme_ticker, theme_name in THEMES.items():

        try:
            theme_ret = close[theme_ticker].pct_change().dropna()

            s_ret, t_ret = stock_ret.align(theme_ret, join="inner")

            train_s = s_ret.loc[TRAIN_START:TRAIN_END]
            train_t = t_ret.loc[TRAIN_START:TRAIN_END]

            event_s = s_ret.loc[EVENT_START:EVENT_END]
            event_t = t_ret.loc[EVENT_START:EVENT_END]

            if len(train_s) < 200 or len(event_s) < 5:
                continue

            # ===== Regression =====
            X = sm.add_constant(train_t)
            model = sm.OLS(train_s, X).fit()

            expected = model.predict(sm.add_constant(event_t))
            car = (event_s - expected).cumsum()
            final_car = car.iloc[-1] * 100

            # ===== Indikatoren =====
            delta = close[stock].diff()
            gain = delta.clip(lower=0).rolling(14).mean()
            loss = -delta.clip(upper=0).rolling(14).mean()
            rs = gain / loss
            rsi = 100 - (100 / (1 + rs))

            sma200 = close[stock].rolling(200).mean()
            dist_sma = (close[stock] / sma200 - 1) * 100

            ema12 = close[stock].ewm(span=12).mean()
            ema26 = close[stock].ewm(span=26).mean()
            macd_hist = ema12 - ema26

            rel_perf = (
                event_s.add(1).prod() -
                event_t.add(1).prod()
            ) * 100

            # ===== Scoring =====
            score = 0

            if final_car < -5: score += 1
            if rsi.iloc[-1] < 35: score += 1
            if dist_sma.iloc[-1] < -10: score += 1
            if macd_hist.iloc[-1] > 0: score += 1
            if rel_perf < -5: score += 1

            if score >= MIN_SCORE:
                if stock not in results:
                    results[stock] = {}
                results[stock][theme_name] = score

        except:
            continue

# ===================================
# OUTPUT
# ===================================

print("="*70)
print(f"MULTI-THEME SCREENING RESULTS (Score >= {MIN_SCORE})")
print("="*70)

for stock, themes in results.items():
    print(f"\n {stock}")
    for theme, score in themes.items():
        print(f"   → {theme} | Score: {score}")

MULTI-THEME SCREENING RESULTS (Score >= 4)

 ADP
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → Artificial_Intelligence | Score: 4

 CRWD
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → Artificial_Intelligence | Score: 4
   → Disruptive_Innovation | Score: 4
   → Cloud_Computing | Score: 4

 INTU
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → Artificial_Intelligence | Score: 4
   → Disruptive_Innovation | Score: 4
   → Cloud_Computing | Score: 4

 MELI
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → Artificial_Intelligence | Score: 4

 PANW
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → Artificial_Intelligence | Score: 4
   → Disruptive_Innovation | Score: 4
   → Cloud_Computing | Score: 4

 TEAM
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → Artificial_Intelligence | Score: 4
   → Disruptive_Innovation | Score: 4
   → Cloud_Computing | Score: 4

 ZS
   → BigTech | Score: 4
   → Semiconductors | Score: 4
   → A